### Dependencies

In [1]:
!pip install -q transformers torch

### Imports

In [2]:
from transformers import pipeline
import pandas as pd
from pprint import pprint

### Load toxicity model

In [3]:
MODEL_NAME = "unitary/toxic-bert"

classifier = pipeline(
    task="text-classification",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    top_k=None
)

print("Model loaded:", MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded: unitary/toxic-bert


### Helpers

In [4]:
def normalize_label(label: str) -> str:
    return label.strip().lower()


def scores_to_dict(model_output):
    """
    Expected output format:
    [[{'label': 'toxic', 'score': 0.99}, ...]]
    """
    if not model_output:
        return {}

    if isinstance(model_output[0], list):
        items = model_output[0]
    else:
        items = model_output

    return {
        normalize_label(item["label"]): float(item["score"])
        for item in items
    }


def predict_message(message: str):
    output = classifier(message)
    return scores_to_dict(output)


def build_context_text(context_messages, current_message, sep=" [SEP] "):
    cleaned_context = [msg.strip() for msg in context_messages if msg and msg.strip()]
    current_message = current_message.strip()

    if cleaned_context:
        return sep.join(cleaned_context + [current_message])
    return current_message


def predict_with_context(context_messages, current_message, sep=" [SEP] "):
    combined_text = build_context_text(context_messages, current_message, sep=sep)
    scores = predict_message(combined_text)
    return {
        "combined_text": combined_text,
        "scores": scores
    }


def get_primary_toxicity_score(scores: dict) -> float:
    return scores.get("toxic", 0.0)


def classify_severity(scores: dict,
                      severe_threshold=0.90,
                      high_threshold=0.75,
                      medium_threshold=0.50):
    toxic_score = get_primary_toxicity_score(scores)

    if toxic_score >= severe_threshold:
        return "severe"
    elif toxic_score >= high_threshold:
        return "high"
    elif toxic_score >= medium_threshold:
        return "medium"
    else:
        return "low"

### Baseline test

In [5]:
message = "you are trash at this game"

scores = predict_message(message)

print("Message:")
print(message)
print("\nScores:")
pprint(scores)
print("\nSeverity:", classify_severity(scores))

Message:
you are trash at this game

Scores:
{'identity_hate': 0.005062992684543133,
 'insult': 0.8096671104431152,
 'obscene': 0.23836833238601685,
 'severe_toxic': 0.006465201266109943,
 'threat': 0.0008378759957849979,
 'toxic': 0.9530662298202515}

Severity: severe


### Context aware test

In [6]:
context = [
    "you're garbage at this game",
    "i'm not even trying"
]

current_message = "shut up man"

result = predict_with_context(context, current_message)

print("Combined input:")
print(result["combined_text"])
print("\nScores:")
pprint(result["scores"])
print("\nSeverity:", classify_severity(result["scores"]))

Combined input:
you're garbage at this game [SEP] i'm not even trying [SEP] shut up man

Scores:
{'identity_hate': 0.003856110852211714,
 'insult': 0.6805251836776733,
 'obscene': 0.19154678285121918,
 'severe_toxic': 0.0044433861039578915,
 'threat': 0.0009534300770610571,
 'toxic': 0.9511203765869141}

Severity: severe


### With context vs without

In [7]:
def compare_no_context_vs_context(context_messages, current_message):
    baseline_scores = predict_message(current_message)
    context_result = predict_with_context(context_messages, current_message)
    context_scores = context_result["scores"]

    comparison = {
        "message": current_message,
        "context": context_messages,
        "baseline_toxic": get_primary_toxicity_score(baseline_scores),
        "context_toxic": get_primary_toxicity_score(context_scores),
        "baseline_scores": baseline_scores,
        "context_scores": context_scores,
        "baseline_severity": classify_severity(baseline_scores),
        "context_severity": classify_severity(context_scores),
        "combined_text": context_result["combined_text"]
    }
    return comparison

In [8]:
example = compare_no_context_vs_context(
    context_messages=[
        "you're so bad lol",
        "nah bro i'm chilling"
    ],
    current_message="idiot"
)

pprint(example)

{'baseline_scores': {'identity_hate': 0.015480505302548409,
                     'insult': 0.9556223750114441,
                     'obscene': 0.8159763216972351,
                     'severe_toxic': 0.05008123070001602,
                     'threat': 0.0010846953373402357,
                     'toxic': 0.984983503818512},
 'baseline_severity': 'severe',
 'baseline_toxic': 0.984983503818512,
 'combined_text': "you're so bad lol [SEP] nah bro i'm chilling [SEP] idiot",
 'context': ["you're so bad lol", "nah bro i'm chilling"],
 'context_scores': {'identity_hate': 0.005286908242851496,
                    'insult': 0.7189884781837463,
                    'obscene': 0.4245183765888214,
                    'severe_toxic': 0.006709346082061529,
                    'threat': 0.0006674709729850292,
                    'toxic': 0.9149969816207886},
 'context_severity': 'severe',
 'context_toxic': 0.9149969816207886,
 'message': 'idiot'}


### Demo data

In [9]:
demo_data = [
    {
        "context": [],
        "message": "have a nice day",
        "label": "non_toxic"
    },
    {
        "context": [],
        "message": "you are an idiot",
        "label": "toxic"
    },
    {
        "context": ["good game bro", "lmao you sold that round"],
        "message": "you're trash",
        "label": "banter_or_ambiguous"
    },
    {
        "context": ["you're garbage at this game", "i'm not even trying"],
        "message": "shut up man",
        "label": "toxic"
    },
    {
        "context": ["that was actually funny", "nah you're good"],
        "message": "idiot lol",
        "label": "banter_or_ambiguous"
    },
    {
        "context": [],
        "message": "kill yourself",
        "label": "severe_toxic"
    },
    {
        "context": ["check out my stream", "check out my stream", "check out my stream"],
        "message": "check out my stream at mysite.com",
        "label": "spam_or_toxicity_unclear"
    }
]

print(f"Loaded {len(demo_data)} demo examples.")

Loaded 7 demo examples.


### With context vs without (on demo data)

In [10]:
rows = []

for ex in demo_data:
    baseline_scores = predict_message(ex["message"])
    context_result = predict_with_context(ex["context"], ex["message"])
    context_scores = context_result["scores"]

    rows.append({
        "label": ex["label"],
        "message": ex["message"],
        "context": " | ".join(ex["context"]) if ex["context"] else "",
        "baseline_toxic": round(get_primary_toxicity_score(baseline_scores), 4),
        "context_toxic": round(get_primary_toxicity_score(context_scores), 4),
        "baseline_severity": classify_severity(baseline_scores),
        "context_severity": classify_severity(context_scores),
        "baseline_insult": round(baseline_scores.get("insult", 0.0), 4),
        "context_insult": round(context_scores.get("insult", 0.0), 4),
        "baseline_threat": round(baseline_scores.get("threat", 0.0), 4),
        "context_threat": round(context_scores.get("threat", 0.0), 4),
    })

results_df = pd.DataFrame(rows)
results_df

,label,message,context,baseline_toxic,context_toxic,baseline_severity,context_severity,baseline_insult,context_insult,baseline_threat,context_threat
0,non_toxic,have a nice day,,0.0027,0.0027,low,low,0.0002,0.0002,0.0001,0.0001
1,toxic,you are an idiot,,0.9861,0.9861,severe,severe,0.9600,0.9600,0.0013,0.0013
2,banter_or_ambiguous,you're trash,good game bro | lmao you sold that round,0.9794,0.8496,severe,high,0.9111,0.5729,0.0009,0.0005
3,toxic,shut up man,you're garbage at this game | i'm not even trying,0.9390,0.9511,severe,severe,0.1175,0.6805,0.0050,0.0010
4,banter_or_ambiguous,idiot lol,that was actually funny | nah you're good,0.9616,0.8025,severe,high,0.8851,0.6335,0.0006,0.0005
5,severe_toxic,kill yourself,,0.9122,0.9122,severe,severe,0.1386,0.1386,0.8226,0.8226
6,spam_or_toxicity_unclear,check out my stream at mysite.com,check out my stream | check out my stream | ch...,0.0009,0.0012,low,low,0.0002,0.0002,0.0001,0.0001


### Sort by toxicity score change

In [11]:
results_df["toxic_diff"] = (results_df["context_toxic"] - results_df["baseline_toxic"]).abs()
results_df.sort_values("toxic_diff", ascending=False).reset_index(drop=True)

,label,message,context,baseline_toxic,context_toxic,baseline_severity,context_severity,baseline_insult,context_insult,baseline_threat,context_threat,toxic_diff
0,banter_or_ambiguous,idiot lol,that was actually funny | nah you're good,0.9616,0.8025,severe,high,0.8851,0.6335,0.0006,0.0005,0.1591
1,banter_or_ambiguous,you're trash,good game bro | lmao you sold that round,0.9794,0.8496,severe,high,0.9111,0.5729,0.0009,0.0005,0.1298
2,toxic,shut up man,you're garbage at this game | i'm not even trying,0.9390,0.9511,severe,severe,0.1175,0.6805,0.0050,0.0010,0.0121
3,spam_or_toxicity_unclear,check out my stream at mysite.com,check out my stream | check out my stream | ch...,0.0009,0.0012,low,low,0.0002,0.0002,0.0001,0.0001,0.0003
4,non_toxic,have a nice day,,0.0027,0.0027,low,low,0.0002,0.0002,0.0001,0.0001,0.0000
5,toxic,you are an idiot,,0.9861,0.9861,severe,severe,0.9600,0.9600,0.0013,0.0013,0.0000
6,severe_toxic,kill yourself,,0.9122,0.9122,severe,severe,0.1386,0.1386,0.8226,0.8226,0.0000


### Testing different context window sizes

In [12]:
def predict_with_k_context(all_previous_messages, current_message, k, sep=" [SEP] "):
    context_subset = all_previous_messages[-k:] if k > 0 else []
    return predict_with_context(context_subset, current_message, sep=sep)


sample_history = [
    "bro you sold",
    "nah i was lagging",
    "sure lol",
    "you always say that",
    "whatever man"
]

current_message = "idiot"

for k in [0, 1, 3, 5]:
    if k == 0:
        scores = predict_message(current_message)
        combined_text = current_message
    else:
        out = predict_with_k_context(sample_history, current_message, k)
        scores = out["scores"]
        combined_text = out["combined_text"]

    print("=" * 80)
    print(f"k = {k}")
    print("Input:")
    print(combined_text)
    print("toxic score:", round(get_primary_toxicity_score(scores), 4))
    print("severity:", classify_severity(scores))

k = 0
Input:
idiot
toxic score: 0.985
severity: severe
k = 1
Input:
whatever man [SEP] idiot
toxic score: 0.9408
severity: severe
k = 3
Input:
sure lol [SEP] you always say that [SEP] whatever man [SEP] idiot
toxic score: 0.7672
severity: high
k = 5
Input:
bro you sold [SEP] nah i was lagging [SEP] sure lol [SEP] you always say that [SEP] whatever man [SEP] idiot
toxic score: 0.7179
severity: medium
